# E-Commerce Risk & Demand Intelligence System

## Complete End-to-End Engineering Project Blueprint and Practical Workbook

**Project goal:** Turn this current 25,000+-row e-commerce dataset into a complete engineering project that engineered the raw CSV data to analysis, machine learning, model validation, API development, database logging, frontend integration, deployment, documentation, and post-deployment monitoring.

---

## How to use this workbook

1. Read the business question.
2. Read the engineering objective.
3. Write your answer or hypothesis **before** running code.
4. Implement the task.
5. Inspect the output.
6. Explain what the output means.
7. Record the decision you would make.
8. Record what could make your conclusion wrong.
9. Then continue.

The objective is to build models for End-End deployment:

> **Business problem → data → assumptions → experiment → evidence → decision → implementation → deployment → monitoring.**

---

## Current project dataset

Expected file:

`ecommerce_dataset.csv`

---

# PART 0 — PROJECT IDEOLOGY

## The philosophy of this project

### Rule 1: Never begin with the algorithm

First ask:

> What business decision are we trying to improve?

A model is only useful if someone can act on its output.

### Rule 2: A correct-looking answer can still be wrong

A model can achieve a high metric because of:

- leakage;
- duplicates;
- target imbalance;
- unrealistic train/test splitting;
- a target that is mathematically determined by the inputs;
- data that would not exist at prediction time.

### Rule 3: Baselines come before complex models

A complex model is not automatically good. Every model must beat a simple baseline.

### Rule 4: Features must exist at prediction time

If the app predicts cancellation when an order is placed, you may only use information available when the order is placed.

### Rule 5: Deployment is not the finish line

After deployment, ask:

- Are requests valid?
- Are predictions being logged?
- Is the model receiving data similar to training data?
- Are failures handled?
- Can the model be reproduced?
- Can the API be rolled back?

---

# PART 1 — DEFINE THE BUSINESS SYSTEM

## Project name

**E-Commerce Risk & Demand Intelligence System**

The system has two candidate ML capabilities:

### Model A — Cancellation Risk

Business question:

> Given the information available when an order is created, what is the probability that the order will later be cancelled?

Potential decision:

- normal processing;
- manual verification;
- additional customer confirmation.

### Model B — Quantity / Demand Prediction

Business question:

> Given product, location, price, discount, and time-related information, what quantity is expected?

Important distinction:

**Transaction-level quantity prediction is not automatically the same as true demand forecasting.**

---

## Your first written exercise

1. Who uses Model A?
2. What action will they take after a high-risk prediction?
3. What is the cost of a false positive?
4. What is the cost of a false negative?
5. Who uses Model B?
6. What action will they take after a demand prediction?
7. What could go wrong if the quantity prediction is wrong?

Do not continue until you can answer these in plain English.


### Model A — Order Cancellation Prediction

**1. Who uses Model A?**

The e-commerce operations or customer-support team.

**2. What action will they take after a high-risk prediction?**

They can flag the order for review or an established customer/order verification workflow before normal fulfillment. The model provides a risk probability; it does not automatically cancel the order.

**3. What is the cost of a false positive?**

A legitimate order is incorrectly flagged as high risk. This can cause unnecessary manual review or verification, which uses staff time and may create unnecessary friction for a customer.

**4. What is the cost of a false negative?**

An order that is actually going to be cancelled is predicted as low risk and is not flagged. The business may therefore miss an opportunity to review or intervene on that order before cancellation.

### Model B — Quantity/Demand Prediction

**5. Who uses Model B?**

The inventory and purchasing/fulfillment team.

**6. What action will they take after a demand prediction?**

They can use the predicted quantity to help plan inventory, purchasing, and fulfillment capacity.

**7. What could go wrong if the quantity prediction is wrong?**

If demand is overestimated, the business may prepare or purchase more inventory than needed, increasing excess inventory. If demand is underestimated, it may prepare too little inventory, which can contribute to stock shortages and difficulty fulfilling orders.

In [3]:
# Imports and reproducibility setup
from pathlib import Path
import json
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")



df = pd.read_csv('../data/nigeria_ecommerce_25000.csv')

print("Shape:", df.shape)
display(df.head())

ModuleNotFoundError: No module named 'matplotlib'

# PART 2 — DATA UNDERSTANDING BEFORE CLEANING

## Why this comes before cleaning

Do not immediately transform the data.

First understand what exists. Cleaning without understanding can destroy useful evidence.

You must answer:

- What does one row represent?
- Is one row one order, one order line, one customer event, or something else?
- Is `order_id` unique?
- Can one customer appear multiple times?
- Can the same product appear across multiple orders?
- What is the date range?
- Are the numeric values plausible?
- Are categories spelled consistently?

## Challenge 2.1 — Data contract

Write a small data dictionary for every column:

| Column | Meaning | Expected type | Business rule | Potential ML use |
| ------ | ------- | ------------- | ------------- | ---------------- |

Do this before looking at the solutions.


In [ ]:
# 2.1 Basic structure
print(df_raw.shape)
print("\nData types:")
print(df_raw.dtypes)

print("\nMissing values:")
display(df_raw.isna().sum().to_frame("missing_count"))

print("\nDuplicate rows:", df_raw.duplicated().sum())

print("\nUnique values by column:")
display(df_raw.nunique().sort_values().to_frame("n_unique"))

print("\nSummary statistics:")
display(df_raw.describe(include="all").T)

# PART 3 — DATA QUALITY AUDIT

## You are now acting as a data engineer and analyst

Do not ask only:

> Is there a missing value?

Ask:

> Does this value make business sense?

Perform the following audit.

### Challenge 3.1 — Identifier integrity

Check:

- duplicate `order_id`;
- duplicate `customer_id` is not necessarily bad;
- whether one `order_id` maps to multiple rows.

### Challenge 3.2 — Date integrity

Convert `order_date` to datetime and check:

- minimum date;
- maximum date;
- number of unique dates;
- invalid dates.

### Challenge 3.3 — Numeric integrity

Investigate:

- quantity <= 0;
- unit_price <= 0;
- discount outside 0–100;
- total_amount < 0.

### Challenge 3.4 — Categorical integrity

Inspect:

- city;
- category;
- product;
- payment_method;
- order_status.

Look for:

- spelling variants;
- accidental whitespace;
- unexpected labels.

### Challenge 3.5 — Formula audit

Investigate whether `total_amount` is approximately or exactly determined by:

`quantity`, `unit_price`, and `discount_percent`.

This is crucial because a model predicting a mathematically derived target can appear excellent while learning almost nothing useful.


In [ ]:
# 3.1 Identifier integrity
print("order_id duplicates:", df_raw["order_id"].duplicated().sum())
print("customer_id duplicates:", df_raw["customer_id"].duplicated().sum())

order_row_counts = df_raw.groupby("order_id").size().sort_values(ascending=False)
display(order_row_counts.head(10).to_frame("rows_per_order_id"))

# 3.2 Date integrity
df = df_raw.copy()
df["order_date_parsed"] = pd.to_datetime(df["order_date"], errors="coerce")

print("Invalid dates:", df["order_date_parsed"].isna().sum())
print("Minimum date:", df["order_date_parsed"].min())
print("Maximum date:", df["order_date_parsed"].max())
print("Unique dates:", df["order_date_parsed"].nunique())

# 3.3 Numeric integrity
checks = {
    "quantity <= 0": (df["quantity"] <= 0).sum(),
    "unit_price <= 0": (df["unit_price"] <= 0).sum(),
    "discount < 0": (df["discount_percent"] < 0).sum(),
    "discount > 100": (df["discount_percent"] > 100).sum(),
    "total_amount < 0": (df["total_amount"] < 0).sum(),
}
display(pd.Series(checks, name="invalid_count").to_frame())

# 3.4 Categorical integrity
for col in ["city", "category", "product", "payment_method", "order_status"]:
    print(f"\n--- {col} ---")
    display(df[col].astype(str).value_counts(dropna=False).to_frame("count"))

In [ ]:
# 3.5 Formula audit for total_amount
# Hypothesis: total_amount may be related to quantity * unit_price * (1 - discount/100)

df["calculated_amount"] = (
    df["quantity"] * df["unit_price"] * (1 - df["discount_percent"] / 100)
)

df["amount_difference"] = df["total_amount"] - df["calculated_amount"]

display(
    df[[
        "quantity", "unit_price", "discount_percent",
        "total_amount", "calculated_amount", "amount_difference"
    ]].head(20)
)

print("Maximum absolute difference:",
      df["amount_difference"].abs().max())

print("Rows matching within 0.01:",
      (df["amount_difference"].abs() <= 0.01).sum(),
      "out of",
      len(df))

# PART 4 — CLEANING AND TRANSFORMATION

## Do not delete rows automatically

For every cleaning action, document:

1. What problem exists?
2. How many rows are affected?
3. What are the possible choices?
4. Why did you choose this action?
5. What information could be lost?

Create a **Data Cleaning Decision Log**.

Example:

| Problem | Evidence | Action | Why | Rows affected |
| ------- | -------- | ------ | --- | ------------- |

### Cleaning rules to investigate

- duplicate records;
- invalid dates;
- impossible numeric values;
- whitespace in categories;
- inconsistent case;
- missing values;
- outliers.

**Important:** Outliers are not automatically errors. A very expensive order may be a legitimate business event.


In [ ]:
# Start a clean working copy only after documenting decisions.
df = df_raw.copy()

# Example categorical cleanup
categorical_cols = [
    "product", "category", "city",
    "payment_method", "order_status"
]

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

# Parse date
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

# Verify
print(df.dtypes)
print(df.isna().sum())

# PART 5 — FEATURE ENGINEERING

## Why feature engineering exists

Raw columns are not always the best representation of the business problem.

A date can become:

- year;
- month;
- day;
- day of week;
- weekend indicator.

A transaction can become:

- gross amount;
- discount amount;
- price band.

But every feature must pass two tests:

### Test A — Business logic

Can you explain why the feature might help?

### Test B — Prediction-time availability

Would this information exist when the prediction is requested?

If the answer to Test B is no, do not use it for that prediction scenario.

## Challenge 5.1 — Build and justify features

For every feature you create, write:

> I created this feature because...

Then write:

> This feature is available at prediction time because...

Suggested features:

- `order_year`
- `order_month`
- `order_day`
- `day_of_week`
- `is_weekend`
- `gross_amount`
- `discount_amount`
- `price_after_discount`
- `unit_price_band`

Do not automatically use all of them in every model.


In [ ]:
# 5.1 Date features
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_day"] = df["order_date"].dt.day
df["day_of_week"] = df["order_date"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Monetary features
df["gross_amount"] = df["quantity"] * df["unit_price"]
df["discount_amount"] = (
    df["gross_amount"] * df["discount_percent"] / 100
)
df["price_after_discount"] = (
    df["gross_amount"] - df["discount_amount"]
)

# Price bands are interpretable for analysis.
df["unit_price_band"] = pd.cut(
    df["unit_price"],
    bins=4,
    include_lowest=True
)

display(df.head())

# PART 6 — EXPLORATORY DATA ANALYSIS

## The purpose of EDA

Do not make charts because charts look professional.

Every chart must answer a question.

For every visualization:

1. What question does it answer?
2. What pattern do you observe?
3. Could the pattern be caused by sample size?
4. What business action might follow?
5. What additional data would you want before acting?

---

## Required EDA questions

### Revenue and sales

1. What is total revenue?
2. What is average order value?
3. Which products generate the most revenue?
4. Which categories generate the most revenue?
5. Which cities generate the most revenue?
6. Which payment methods are associated with the most revenue?

### Quantity

7. Which products sell the highest total quantity?
8. Which categories have the highest average quantity?
9. Does discount appear related to quantity?

### Time

10. How does revenue change over time?
11. How does quantity change over time?
12. Which days of the week are strongest?
13. Are weekends different?

### Cancellation

14. What is the class distribution of `order_status`?
15. Which city has the highest cancellation rate?
16. Which payment method has the highest cancellation rate?
17. Which category has the highest cancellation rate?
18. Does discount appear related to cancellation?

### Customer

19. How many unique customers exist?
20. Who are the highest-value customers?
21. How many repeat customers exist?
22. What is the distribution of orders per customer?

### Outliers

23. Which transactions are unusually large?
24. Are they errors or legitimate transactions?


In [ ]:
# Helper functions for clean charts
def plot_bar(series, title, xlabel="", ylabel="Value", rotation=45):
    ax = series.plot(kind="bar", figsize=(10, 5))
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    plt.show()

def plot_line(series, title, xlabel="", ylabel="Value"):
    ax = series.plot(figsize=(10, 5), marker="o")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    plt.show()

# Revenue by product
revenue_by_product = (
    df.groupby("product")["total_amount"]
      .sum()
      .sort_values(ascending=False)
)
plot_bar(revenue_by_product, "Total Revenue by Product", "Product", "Revenue")

# Revenue by category
revenue_by_category = (
    df.groupby("category")["total_amount"]
      .sum()
      .sort_values(ascending=False)
)
plot_bar(revenue_by_category, "Total Revenue by Category", "Category", "Revenue")

# Revenue by city
revenue_by_city = (
    df.groupby("city")["total_amount"]
      .sum()
      .sort_values(ascending=False)
)
plot_bar(revenue_by_city, "Total Revenue by City", "City", "Revenue")

# Quantity by product
quantity_by_product = (
    df.groupby("product")["quantity"]
      .sum()
      .sort_values(ascending=False)
)
plot_bar(quantity_by_product, "Total Quantity by Product", "Product", "Units")

# Revenue over time
daily_revenue = (
    df.groupby("order_date")["total_amount"]
      .sum()
      .sort_index()
)
plot_line(daily_revenue, "Revenue Over Time", "Date", "Revenue")

In [ ]:
# Cancellation analysis
status_counts = df["order_status"].value_counts(dropna=False)
display(status_counts.to_frame("count"))

status_rates = df["order_status"].value_counts(normalize=True, dropna=False)
display((status_rates * 100).round(2).to_frame("percent"))

# Binary cancellation indicator for analysis
df["is_cancelled"] = (
    df["order_status"].astype(str).str.lower() == "cancelled"
).astype(int)

for col in ["city", "payment_method", "category"]:
    cancellation_rate = (
        df.groupby(col)["is_cancelled"]
          .agg(["mean", "count"])
          .sort_values("mean", ascending=False)
    )
    cancellation_rate["cancellation_rate_percent"] = (
        cancellation_rate["mean"] * 100
    )
    print(f"\nCancellation analysis by {col}")
    display(cancellation_rate)
    plot_bar(
        cancellation_rate["cancellation_rate_percent"],
        f"Cancellation Rate by {col}",
        col,
        "Cancellation Rate (%)"
    )

# PART 7 — BUSINESS INSIGHT REPORT

You are now required to stop coding and think like an analyst.

Create a section titled:

## Executive Findings

Write at least:

### Finding 1 — Revenue

- Evidence:
- Interpretation:
- Proposed action:
- Confidence:
- Limitation:

### Finding 2 — Product/category

- Evidence:
- Interpretation:
- Proposed action:
- Confidence:
- Limitation:

### Finding 3 — City

- Evidence:
- Interpretation:
- Proposed action:
- Confidence:
- Limitation:

### Finding 4 — Cancellation

- Evidence:
- Interpretation:
- Proposed action:
- Confidence:
- Limitation:

### Finding 5 — Customer behavior

- Evidence:
- Interpretation:
- Proposed action:
- Confidence:
- Limitation:

Do not write conclusions such as:

> Lagos causes cancellations.

Observational data usually supports association, not automatic causation.


# PART 8 — ML PROBLEM 1: CANCELLATION RISK CLASSIFICATION

## The exact prediction contract

### Prediction moment

The prediction is requested **when the order is created**.

### Target

`Cancelled` versus `Delivered`.

### Business output

A probability:

`P(order will be cancelled | information available at order creation)`

### Possible action

Use the probability to prioritize verification.

---

## Leakage audit

For every candidate feature, create this table:

| Feature | Available at order creation? | Why? | Use? |
| ------- | ---------------------------- | ---- | ---- |

Candidate features:

- order_id
- customer_id
- product
- category
- quantity
- unit_price
- discount_percent
- total_amount
- city
- payment_method
- order_date-derived features
- gross_amount
- discount_amount
- price_after_discount

### Important

Do not use `order_status` itself as a feature.

Do not use any feature created after the order outcome is known.

Be cautious with:

- `total_amount`;
- `price_after_discount`;
- engineered features that are deterministic duplicates of other features.

Your first model should use a deliberately clean, defensible feature set.


In [ ]:
# Prepare classification dataset
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Restrict to the two states only if both exist.
clf_df = df[
    df["order_status"].astype(str).str.lower().isin(
        ["cancelled", "delivered"]
    )
].copy()

clf_df["target_cancelled"] = (
    clf_df["order_status"].astype(str).str.lower() == "cancelled"
).astype(int)

print(clf_df["target_cancelled"].value_counts())
print(clf_df["target_cancelled"].value_counts(normalize=True))

# Initial conservative feature list.
classification_features = [
    "product",
    "category",
    "quantity",
    "unit_price",
    "discount_percent",
    "city",
    "payment_method",
    "order_month",
    "day_of_week",
    "is_weekend",
]

X_clf = clf_df[classification_features].copy()
y_clf = clf_df["target_cancelled"].copy()

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

categorical_features = [
    "product", "category", "city", "payment_method"
]

numeric_features = [
    "quantity", "unit_price", "discount_percent",
    "order_month", "day_of_week", "is_weekend"
]

preprocessor_clf = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

print("Training rows:", len(X_train_clf))
print("Test rows:", len(X_test_clf))

## Challenge 8.1 — Baseline before intelligence

Train a `DummyClassifier`.

Question:

> What score can we obtain without learning meaningful patterns?

If your sophisticated model barely beats the baseline, that is an important result.

Do not hide disappointing results.


In [ ]:
# Baseline pipeline
dummy_pipeline = Pipeline([
    ("preprocessor", preprocessor_clf),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_pipeline.fit(X_train_clf, y_train_clf)
dummy_pred = dummy_pipeline.predict(X_test_clf)

print("Dummy accuracy:",
      accuracy_score(y_test_clf, dummy_pred))

print("Dummy F1:",
      f1_score(y_test_clf, dummy_pred, zero_division=0))

## Challenge 8.2 — Train at least two real classifiers

Start with:

1. Logistic Regression;
2. Random Forest.

Why?

Logistic Regression gives you a simpler baseline.
Random Forest can capture non-linear interactions.

Do not conclude that Random Forest is better just because it is more complex.

Evaluate:

- accuracy;
- precision;
- recall;
- F1;
- ROC-AUC;
- precision-recall curve;
- confusion matrix.

### Your written reasoning

Which metric matters most for this business problem?

If you choose high recall:

- what happens to false positives?

If you choose high precision:

- what happens to false negatives?

There is no universal correct answer. The answer depends on the business cost.


In [ ]:
# Logistic Regression pipeline
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor_clf),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

logreg_pipeline.fit(X_train_clf, y_train_clf)

# Random Forest pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor_clf),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

rf_pipeline.fit(X_train_clf, y_train_clf)

def evaluate_classifier(model, X_test, y_test, name):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    results = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba),
    }

    print(results)
    print("\nClassification report:")
    print(classification_report(y_test, pred, zero_division=0))

    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

    RocCurveDisplay.from_predictions(y_test, proba)
    plt.title(f"ROC Curve — {name}")
    plt.show()

    PrecisionRecallDisplay.from_predictions(y_test, proba)
    plt.title(f"Precision-Recall Curve — {name}")
    plt.show()

    return results

clf_results = []
clf_results.append(
    evaluate_classifier(
        logreg_pipeline,
        X_test_clf,
        y_test_clf,
        "Logistic Regression"
    )
)
clf_results.append(
    evaluate_classifier(
        rf_pipeline,
        X_test_clf,
        y_test_clf,
        "Random Forest"
    )
)

display(pd.DataFrame(clf_results).sort_values("f1", ascending=False))

# PART 9 — THRESHOLD ENGINEERING

A probability model does not force you to use 0.50 as the decision threshold.

Suppose the model outputs:

`0.18`

Should you flag the order?

That depends on the cost of mistakes.

Test thresholds such as:

- 0.20;
- 0.30;
- 0.40;
- 0.50;
- 0.60.

For each threshold calculate:

- precision;
- recall;
- F1;
- number of orders flagged.

Then explain:

> Which threshold would the operations team choose and why?

This is a real engineering/business decision, not merely a model setting.


In [ ]:
# Threshold study
from sklearn.metrics import precision_score, recall_score, f1_score

best_classifier = rf_pipeline  # Replace after your comparison if needed.

proba = best_classifier.predict_proba(X_test_clf)[:, 1]

threshold_rows = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    pred_threshold = (proba >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_test_clf, pred_threshold, zero_division=0
        ),
        "recall": recall_score(
            y_test_clf, pred_threshold, zero_division=0
        ),
        "f1": f1_score(
            y_test_clf, pred_threshold, zero_division=0
        ),
        "orders_flagged": int(pred_threshold.sum())
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results)

# PART 10 — ML PROBLEM 2: QUANTITY / DEMAND PREDICTION

## First, validate the framing

Do not automatically call this forecasting.

### Stage 1

Build a supervised regression problem:

> Predict transaction quantity.

### Stage 2

Investigate whether the data can be aggregated into a meaningful time-based demand dataset.

For example:

`date + product + city → total quantity`

Only call the final system a demand forecaster if:

- dates cover enough time;
- there are enough repeated observations;
- the prediction is genuinely about a future period;
- the train/test split respects time.

---

## Critical leakage question

If predicting quantity at order creation, can you use:

- total_amount?
- gross_amount?

Usually these are dangerous because they depend directly on quantity.

Do not feed the model variables that mathematically contain the target.

Your initial quantity model should therefore exclude deterministic descendants of `quantity`.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

regression_features = [
    "product",
    "category",
    "unit_price",
    "discount_percent",
    "city",
    "payment_method",
    "order_month",
    "day_of_week",
    "is_weekend",
]

reg_df = df.dropna(subset=["quantity"]).copy()

X_reg = reg_df[regression_features]
y_reg = reg_df["quantity"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=RANDOM_STATE
)

reg_categorical = [
    "product", "category", "city", "payment_method"
]

reg_numeric = [
    "unit_price", "discount_percent",
    "order_month", "day_of_week", "is_weekend"
]

preprocessor_reg = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            reg_numeric
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            reg_categorical
        )
    ]
)

dummy_regression = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", DummyRegressor(strategy="mean"))
])

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

rf_regression = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE
    ))
])

for model in [
    dummy_regression,
    ridge_pipeline,
    rf_regression
]:
    model.fit(X_train_reg, y_train_reg)

def evaluate_regressor(model, X_test, y_test, name):
    pred = model.predict(X_test)

    results = {
        "model": name,
        "mae": mean_absolute_error(y_test, pred),
        "rmse": mean_squared_error(y_test, pred) ** 0.5,
        "r2": r2_score(y_test, pred)
    }

    print(results)

    plt.figure(figsize=(7, 6))
    plt.scatter(y_test, pred)
    plt.xlabel("Actual Quantity")
    plt.ylabel("Predicted Quantity")
    plt.title(f"Actual vs Predicted — {name}")

    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    plt.plot([min_val, max_val], [min_val, max_val])
    plt.tight_layout()
    plt.show()

    residuals = y_test - pred
    plt.figure(figsize=(8, 5))
    plt.hist(residuals, bins=20)
    plt.title(f"Residual Distribution — {name}")
    plt.xlabel("Actual - Predicted")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    return results

reg_results = []
reg_results.append(
    evaluate_regressor(
        dummy_regression,
        X_test_reg,
        y_test_reg,
        "Dummy Mean Regressor"
    )
)
reg_results.append(
    evaluate_regressor(
        ridge_pipeline,
        X_test_reg,
        y_test_reg,
        "Ridge Regression"
    )
)
reg_results.append(
    evaluate_regressor(
        rf_regression,
        X_test_reg,
        y_test_reg,
        "Random Forest Regressor"
    )
)

display(pd.DataFrame(reg_results).sort_values("mae"))

# PART 11 — DEMAND FORECASTING FEASIBILITY STUDY

Now inspect whether a real future-oriented demand problem is possible.

## Build this table

Aggregate:

`order_date + product + city`

and calculate:

- total quantity;
- number of transactions;
- average unit price;
- average discount.

Then answer:

1. How many unique dates exist?
2. How many dates exist per product?
3. How many dates exist per product-city combination?
4. Are there enough observations for a meaningful time split?
5. Is there enough historical variation?
6. Are there missing time periods?

If the answer is no, **do not fake a forecasting project**.

Instead, honestly document:

> The current dataset supports transaction-level quantity prediction, but the temporal density is insufficient for robust demand forecasting.

That is a stronger engineering conclusion than pretending.


In [ ]:
# Demand aggregation study
demand_daily = (
    df.groupby(["order_date", "product", "city"], as_index=False)
      .agg(
          total_quantity=("quantity", "sum"),
          transactions=("order_id", "count"),
          avg_unit_price=("unit_price", "mean"),
          avg_discount=("discount_percent", "mean")
      )
      .sort_values("order_date")
)

display(demand_daily.head(20))

coverage = (
    demand_daily.groupby(["product", "city"])
    .agg(
        first_date=("order_date", "min"),
        last_date=("order_date", "max"),
        observations=("order_date", "count")
    )
    .sort_values("observations", ascending=False)
)

display(coverage.head(30))

# PART 12 — MODEL VALIDATION AND DEBUGGING

## Your model is not finished when `.fit()` works

You must deliberately try to break your own model.

### Test 1 — Leakage test

Ask:

- Did I accidentally include the target?
- Did I include a descendant of the target?
- Did I preprocess before splitting?
- Did I use information from the future?

### Test 2 — Baseline test

Ask:

- Does the model beat a simple baseline?

### Test 3 — Stability test

Use cross-validation where appropriate.

### Test 4 — Error slicing

Where does the model fail?

- city;
- product;
- category;
- price band;
- discount band.

### Test 5 — Plausibility test

Create realistic manual inputs.

Does the model produce absurd outputs?

### Test 6 — Production schema test

Can the saved model predict when a column is missing?
It should fail clearly, not silently produce nonsense.

---

## Classification error analysis

Create a DataFrame containing:

- actual;
- prediction;
- probability;
- city;
- product;
- category;
- payment_method.

Inspect:

- false positives;
- false negatives.

Ask:

> What do the mistakes have in common?


In [ ]:
# Classification error analysis
clf_pred = best_classifier.predict(X_test_clf)
clf_proba = best_classifier.predict_proba(X_test_clf)[:, 1]

error_analysis_clf = X_test_clf.copy()
error_analysis_clf["actual"] = y_test_clf.values
error_analysis_clf["prediction"] = clf_pred
error_analysis_clf["cancel_probability"] = clf_proba
error_analysis_clf["error_type"] = np.select(
    [
        (error_analysis_clf["actual"] == 1) &
        (error_analysis_clf["prediction"] == 0),
        (error_analysis_clf["actual"] == 0) &
        (error_analysis_clf["prediction"] == 1),
        (error_analysis_clf["actual"] == 1) &
        (error_analysis_clf["prediction"] == 1),
    ],
    [
        "False Negative",
        "False Positive",
        "True Positive",
    ],
    default="True Negative"
)

display(
    error_analysis_clf.sort_values(
        "cancel_probability", ascending=False
    ).head(30)
)

print("False negatives:")
display(
    error_analysis_clf[
        error_analysis_clf["error_type"] == "False Negative"
    ].head(20)
)

print("False positives:")
display(
    error_analysis_clf[
        error_analysis_clf["error_type"] == "False Positive"
    ].head(20)
)

# PART 13 — CROSS-VALIDATION AND MODEL SELECTION

Do not repeatedly tune against the test set.

The test set is your final exam.

Recommended process:

1. Split train/test once.
2. Develop pipelines on training data.
3. Use cross-validation on training data.
4. Choose the final candidate.
5. Evaluate once on the untouched test set.
6. Record the result.

Do not keep changing the model until the test score looks impressive.

That turns the test set into another training signal.


In [ ]:
# Example cross-validation on the training split only
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_scores = cross_val_score(
    best_classifier,
    X_train_clf,
    y_train_clf,
    scoring="f1",
    cv=cv
)

print("CV F1 scores:", cv_scores)
print("Mean CV F1:", cv_scores.mean())
print("Std CV F1:", cv_scores.std())

# PART 14 — FINAL MODEL DECISION DOCUMENT

Before saving anything, complete this checklist.

## Cancellation model

- [ ] I can state the exact prediction moment.
- [ ] Every feature is available at that moment.
- [ ] I checked class balance.
- [ ] I trained a baseline.
- [ ] I compared at least two models.
- [ ] I selected metrics based on business cost.
- [ ] I inspected false positives.
- [ ] I inspected false negatives.
- [ ] I tested decision thresholds.
- [ ] I performed cross-validation.
- [ ] I evaluated the untouched test set.
- [ ] I documented limitations.

## Quantity model

- [ ] I defined whether this is transaction prediction or forecasting.
- [ ] I removed features that contain quantity mathematically.
- [ ] I trained a baseline.
- [ ] I compared at least two models.
- [ ] I inspected residuals.
- [ ] I checked negative predictions.
- [ ] I investigated time coverage.
- [ ] I did not falsely claim forecasting if the data cannot support it.
- [ ] I documented limitations.

Only save a model after these questions are answered.


# PART 15 — SAVE THE COMPLETE PIPELINES

## Why save the pipeline?

Your preprocessing must be identical during:

- training;
- testing;
- API inference;
- frontend requests.

Save the entire fitted pipeline, not only the estimator.

Recommended artifacts:

- `models/cancellation_pipeline.joblib`
- `models/quantity_pipeline.joblib`

Also save:

- metadata;
- feature list;
- package versions;
- evaluation metrics.

Never load an untrusted pickle/joblib artifact.


In [ ]:
import joblib
import sklearn

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# Replace these assignments with your chosen final models.
final_cancellation_pipeline = best_classifier
final_quantity_pipeline = rf_regression

joblib.dump(
    final_cancellation_pipeline,
    MODELS_DIR / "cancellation_pipeline.joblib"
)

joblib.dump(
    final_quantity_pipeline,
    MODELS_DIR / "quantity_pipeline.joblib"
)

metadata = {
    "project": "E-Commerce Risk & Demand Intelligence System",
    "random_state": RANDOM_STATE,
    "classification_features": classification_features,
    "regression_features": regression_features,
    "scikit_learn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}

with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
for path in MODELS_DIR.iterdir():
    print("-", path)

# PART 16 — RELOAD TEST

Saving successfully is not enough.

Reload the models into a fresh variable and test a prediction.

This simulates what your API will do.

Also verify:

- column names;
- column order;
- data types;
- unknown categories.

The production model contract must be explicit.


In [ ]:
loaded_cancellation = joblib.load(
    MODELS_DIR / "cancellation_pipeline.joblib"
)

loaded_quantity = joblib.load(
    MODELS_DIR / "quantity_pipeline.joblib"
)

sample_clf_input = X_test_clf.head(1).copy()
sample_reg_input = X_test_reg.head(1).copy()

print(
    "Cancellation probability:",
    loaded_cancellation.predict_proba(sample_clf_input)[:, 1]
)

print(
    "Predicted quantity:",
    loaded_quantity.predict(sample_reg_input)
)

# PART 17 — PROJECT REPOSITORY STRUCTURE

Create the repository like this:

```text
ecommerce-risk-demand-intelligence/
│
├── data/
│   ├── raw/
│   │   └── ecommerce_dataset.csv
│   └── processed/
│
├── notebooks/
│   ├── 01_data_understanding_and_eda.ipynb
│   ├── 02_cancellation_model.ipynb
│   ├── 03_quantity_model.ipynb
│   └── 04_model_validation.ipynb
│
├── src/
│   ├── __init__.py
│   ├── config.py
│   ├── data_validation.py
│   ├── feature_engineering.py
│   ├── train_cancellation.py
│   ├── train_quantity.py
│   └── evaluation.py
│
├── models/
│   ├── cancellation_pipeline.joblib
│   ├── quantity_pipeline.joblib
│   └── model_metadata.json
│
├── backend/
│   ├── main.py
│   ├── schemas.py
│   ├── database.py
│   ├── models.py
│   └── requirements.txt
│
├── frontend/
│   ├── index.html
│   ├── styles.css
│   └── app.js
│
├── tests/
│   ├── test_api.py
│   ├── test_features.py
│   └── test_validation.py
│
├── Dockerfile
├── docker-compose.yml
├── requirements.txt
├── README.md
├── .env.example
└── .gitignore
```

You do not need to build all files immediately.

Build them in the sequence defined below.


# PART 17A — CI/CD WITH GITHUB ACTIONS AND DOCKER

## Why add CI/CD and containerization?

This project should not remain a notebook-only exercise. A strong end-to-end ML project needs:

- automated checks on every push and pull request;
- reproducible environments for local development and deployment;
- consistent dependency installation;
- a containerized backend for deployment reliability;
- a single source of truth for build and test commands.

GitHub Actions and Docker are not optional upgrades. They are part of the professional engineering setup.

---

## Recommended repository additions

Add these items to the project structure:

```text
ecommerce-risk-demand-intelligence/
│
├── .github/
│   └── workflows/
│       └── ci.yml
├── .dockerignore
├── Dockerfile
├── docker-compose.yml
├── data/
│   ├── raw/
│   │   └── ecommerce_dataset.csv
│   └── processed/
│
├── notebooks/
│   ├── 01_data_understanding_and_eda.ipynb
│   ├── 02_cancellation_model.ipynb
│   ├── 03_quantity_model.ipynb
│   └── 04_model_validation.ipynb
│
├── src/
│   ├── __init__.py
│   ├── config.py
│   ├── data_validation.py
│   ├── feature_engineering.py
│   ├── train_cancellation.py
│   ├── train_quantity.py
│   └── evaluation.py
│
├── models/
│   ├── cancellation_pipeline.joblib
│   ├── quantity_pipeline.joblib
│   └── model_metadata.json
│
├── backend/
│   ├── Dockerfile
│   ├── main.py
│   ├── schemas.py
│   ├── database.py
│   ├── models.py
│   └── requirements.txt
│
├── frontend/
│   ├── index.html
│   ├── styles.css
│   └── app.js
│
├── tests/
│   ├── test_api.py
│   ├── test_features.py
│   └── test_validation.py
│
├── requirements.txt
├── README.md
├── .env.example
├── .gitignore
└── .pre-commit-config.yaml
```

---

## GitHub Actions CI/CD checklist

Use GitHub Actions to automate the project lifecycle.

Recommended workflow triggers:

- push to `main`;
- pull request to `main`;
- manual dispatch for deployment checks.

Minimum CI checks:

1. set up Python;
2. install dependencies;
3. run unit tests;
4. run API smoke tests;
5. run lint/format checks if configured;
6. build the Docker image;
7. optionally deploy to Render or another hosting platform after tests pass.

### Example workflow responsibilities

- `test` job: run `pytest` and validate API contracts.
- `build` job: run `docker build -t ecommerce-risk-demand .`.
- `deploy` job: trigger only after successful validation and proper secrets are present.

### Recommended GitHub secrets

Store in the repository settings:

- `RENDER_API_KEY`;
- `RENDER_SERVICE_ID`;
- `DATABASE_URL`;
- any other cloud credentials.

Never commit real secrets into the repository.

---

## Docker strategy

Docker should be used for repeatable local setup and deployment consistency.

### Use cases

- run the FastAPI backend in a consistent environment;
- run a PostgreSQL database with `docker-compose`;
- build an image that can be deployed to Render, Azure Container Apps, or another host;
- verify that the app works outside the notebook environment.

### Recommended Docker setup

- root-level `Dockerfile` for the full app or API image;
- `backend/Dockerfile` if the API is separated from the rest of the project;
- `docker-compose.yml` to define backend, database, and optional frontend services;
- `.dockerignore` to avoid copying unnecessary files into the image.

### Example Docker principles

- install only production dependencies;
- copy the app code and model artifacts;
- expose the correct port;
- run the app with a stable entry point such as:

```bash
uvicorn main:app --host 0.0.0.0 --port 8000
```

---

## CI/CD and Docker decision rules

- Do not rely only on local notebook execution.
- Do not assume that a project is deployment-ready because it runs once on a laptop.
- Do not store secrets in `.env` files that are committed to git.
- Do not skip tests before deployment.
- Do not build Docker images without verifying that the API can start successfully.

This is the professional standard expected for an end-to-end ML portfolio project.


# PART 18 — API ARCHITECTURE

## Recommended deployment architecture for this project

Yes, this architecture is valid:

```text
Browser
   │
   ▼
HTML / CSS / JavaScript Frontend
   │ HTTP / JSON
   ▼
FastAPI Backend
   │
   ├── Validate request
   ├── Load ML pipelines
   ├── Generate predictions
   └── Log prediction transaction
          │
          ▼
      PostgreSQL
          │
          ▼
         Neon
```

The backend can be deployed on Render.

The PostgreSQL database can be hosted on Neon.

### Why this architecture is stronger than a simple Streamlit demo

It teaches you:

- frontend/backend separation;
- HTTP APIs;
- request validation;
- JSON contracts;
- database persistence;
- environment variables;
- CORS;
- deployment;
- production-style project organization.

### Important distinction

For a fast portfolio demo, Streamlit is easier.

For this project, **HTML/JS + FastAPI + PostgreSQL is the stronger engineering route**.

Do not build both frontend architectures at the same time.

Recommended order:

1. Build models.
2. Build FastAPI.
3. Test API with Swagger/Postman/curl.
4. Add PostgreSQL logging.
5. Build HTML/JS frontend.
6. Connect frontend to API.
7. Deploy.


# PART 19 — FASTAPI CONTRACT

## Endpoints to build

### Health check

`GET /health`

Purpose:

- verify that the API is alive.

Response:

```json
{
  "status": "ok"
}
```

### Cancellation prediction

`POST /predict/cancellation`

Input fields must match the trained model contract.

Example conceptual request:

```json
{
  "product": "Smart Watch",
  "category": "Electronics",
  "quantity": 2,
  "unit_price": 50000,
  "discount_percent": 10,
  "city": "Lagos",
  "payment_method": "Card",
  "order_month": 12,
  "day_of_week": 4,
  "is_weekend": 0
}
```

Response should include:

- probability;
- predicted class;
- risk label;
- model version;
- timestamp.

### Quantity prediction

`POST /predict/quantity`

Response should include:

- predicted quantity;
- non-negative validation;
- model version;
- timestamp.

### Prediction history

`GET /predictions`

For development:

- return recent prediction records.

Later:

- add authentication and pagination.

---

## API validation requirements

Use Pydantic models.

Validate:

- quantity > 0;
- unit_price > 0;
- discount_percent between 0 and 100;
- required categorical values are not empty.

Do not let raw frontend data enter your model without validation.


# PART 20 — DATABASE DESIGN

## Why log predictions?

Without logging, you cannot answer:

- Who used the system?
- When was a prediction made?
- What input was supplied?
- What did the model predict?
- Which model version produced it?
- Did the request fail?

## Start with two tables

### Table: `prediction_logs`

Suggested fields:

- `id`
- `prediction_type`
- `created_at`
- `model_version`
- `request_payload`
- `prediction_value`
- `prediction_probability`
- `risk_label`
- `status`

### Optional later table: `feedback`

Purpose:
store actual outcomes after prediction.

Suggested fields:

- `id`
- `prediction_log_id`
- `actual_outcome`
- `recorded_at`

This creates a future path toward monitoring real model performance.

---

## Critical privacy principle

Do not log secrets.

Do not log passwords, tokens, API keys, or unnecessary personal information.

Your current dataset does not include highly detailed personal data, but the principle matters for real systems.


# PART 21 — FRONTEND REQUIREMENTS

Build a simple professional dashboard.

## Page structure

### Header

- project title;
- short explanation.

### Navigation

- Cancellation Risk;
- Quantity Prediction;
- About the Models.

### Cancellation form

Fields matching the API contract.

### Output

Display:

- probability;
- risk level;
- interpretation.

Do not use language such as:

> This order will definitely be cancelled.

Use:

> The model estimates an X% cancellation risk based on patterns in the training data.

### Quantity form

Display:

- predicted quantity;
- interpretation;
- limitation.

### About section

Explain:

- dataset;
- model purpose;
- limitations;
- this is a portfolio/educational project.

---

## Frontend engineering questions

1. What happens while the API request is running?
2. What happens if the API is unavailable?
3. What happens if the server returns validation errors?
4. What happens if the user enters impossible values?
5. How do you show a useful error message without exposing internal stack traces?


# PART 22 — DEPLOYMENT PLAN

## Architecture

```text
GitHub repository
       │
       ├──────────────► Render
       │                 FastAPI backend
       │
       ├──────────────► Static frontend host
       │                 HTML / CSS / JS
       │
       └──────────────► Neon
                         PostgreSQL
```

You may also host the frontend on a platform that supports static sites.

## Render deployment checklist

Before deployment:

- [ ] `requirements.txt` exists.
- [ ] API starts locally.
- [ ] environment variables are not hard-coded.
- [ ] database connection uses an environment variable.
- [ ] CORS is configured correctly.
- [ ] health endpoint works.
- [ ] model files are available to the backend.
- [ ] `.gitignore` excludes secrets.
- [ ] README contains setup instructions.

A typical FastAPI Render start command is:

`uvicorn main:app --host 0.0.0.0 --port $PORT`

## Neon checklist

- [ ] Create database.
- [ ] Obtain connection string.
- [ ] Store it as an environment variable.
- [ ] Never commit the real connection string.
- [ ] Create tables through migrations or controlled schema setup.
- [ ] Test database connection locally before deployment.

---

## Deployment testing

After deployment:

1. Open `/health`.
2. Send a valid cancellation request.
3. Send an invalid request.
4. Send a quantity request.
5. Confirm predictions are logged.
6. Check database rows.
7. Open the frontend.
8. Test the complete user journey.
9. Check backend logs.
10. Document the live URL in README.


# PART 23 — TESTING STRATEGY

A professional project is not tested only by clicking the UI.

## Unit tests

Test:

- feature functions;
- input validation;
- helper functions.

## API tests

Test:

- `/health`;
- valid prediction requests;
- invalid requests;
- missing fields;
- impossible numeric values.

## Integration tests

Test:

Frontend → API → Model → Database

## Regression tests

When you change code:

- do existing predictions still work?
- does the API contract change?
- did a feature column disappear?

## Manual smoke test

Before every major deployment:

- start backend;
- make one valid request;
- make one invalid request;
- confirm database logging;
- inspect logs.


# PART 24 — README: WHAT TO INCLUDE AND WHY

Your README is not decoration.

It should allow another engineer to understand:

- what the project does;
- why it exists;
- how to run it;
- how the models were evaluated;
- what the limitations are.

## Recommended README structure

### 1. Project title

**E-Commerce Risk & Demand Intelligence System**

### 2. One-paragraph overview

Explain the two prediction capabilities and the business decisions they support.

### 3. Architecture

Add a diagram showing:

Frontend → FastAPI → ML Pipelines → PostgreSQL

### 4. Dataset

Document:

- source;
- size;
- columns;
- known limitations.

### 5. Business problems

Explain:

- cancellation risk;
- quantity prediction.

### 6. Methodology

Explain:

- data audit;
- cleaning;
- feature engineering;
- leakage audit;
- train/test strategy;
- baseline models;
- candidate models;
- evaluation metrics.

### 7. Results

Show actual final metrics.

Do not invent impressive numbers.

### 8. How to run locally

Include:

- clone;
- virtual environment;
- install dependencies;
- environment variables;
- start backend;
- start frontend.

### 9. API documentation

List endpoints and example requests.

### 10. Deployment

Explain:

- backend hosting;
- database hosting;
- frontend hosting.

### 11. Limitations

This is essential.

Examples:

- small dataset;
- synthetic or limited data realism;
- uncertain temporal density;
- no real post-deployment feedback yet.

### 12. Future work

Examples:

- larger dataset;
- time-aware demand forecasting;
- model monitoring;
- authentication;
- feedback loop;
- retraining pipeline.

---

## Final README question

Could a stranger clone your repository and understand the project without messaging you?

If no, improve the README.


# PART 25 — FINAL PORTFOLIO DELIVERABLES

You are finished only when these exist.

## Data work

- [ ] data dictionary;
- [ ] cleaning decision log;
- [ ] EDA;
- [ ] visualizations;
- [ ] business findings.

## ML work

- [ ] classification baseline;
- [ ] classification candidate models;
- [ ] threshold analysis;
- [ ] error analysis;
- [ ] regression baseline;
- [ ] regression candidate models;
- [ ] residual analysis;
- [ ] leakage audit;
- [ ] final model decision.

## Engineering

- [ ] saved complete pipelines;
- [ ] metadata;
- [ ] API;
- [ ] request validation;
- [ ] database logging;
- [ ] tests.

## Product

- [ ] HTML/JS frontend;
- [ ] useful loading states;
- [ ] useful error states;
- [ ] clear model output;
- [ ] limitations shown to users.

## Deployment

- [ ] live backend;
- [ ] live database;
- [ ] live frontend;
- [ ] health check;
- [ ] logs;
- [ ] end-to-end test.

## Documentation

- [ ] complete README;
- [ ] architecture diagram;
- [ ] API examples;
- [ ] setup instructions;
- [ ] metrics;
- [ ] limitations;
- [ ] future work.

---

# FINAL PHILOSOPHY

The purpose of this project is not:

> I trained a Random Forest.

The purpose is:

> I started with an ambiguous business problem, interrogated the data, checked whether the prediction problem was valid, prevented leakage, established baselines, evaluated mistakes, made an explicit model decision, packaged the complete preprocessing and model pipeline, exposed it through a validated API, logged inference events, connected a user-facing frontend, deployed the system, and documented its limitations.

That is the mindset you should carry into future AI and ML projects.

---

# YOUR EXECUTION ORDER — DO NOT SKIP AHEAD

## Phase 1

Data understanding and data audit.

## Phase 2

Cleaning decision log.

## Phase 3

Feature engineering and EDA.

## Phase 4

Write business insights.

## Phase 5

Validate cancellation prediction viability.

## Phase 6

Build and evaluate cancellation baseline.

## Phase 7

Build and evaluate real classifiers.

## Phase 8

Perform threshold and error analysis.

## Phase 9

Validate quantity prediction and forecasting feasibility.

## Phase 10

Build regression baselines and candidate models.

## Phase 11

Make final model decisions.

## Phase 12

Save and reload complete pipelines.

## Phase 13

Refactor reusable logic into Python files.

## Phase 14

Build FastAPI.

## Phase 15

Test API before frontend.

## Phase 16

Add PostgreSQL logging.

## Phase 17

Build HTML/JS frontend.

## Phase 18

Perform end-to-end testing.

## Phase 19

Deploy.

## Phase 20

Finish README and project presentation.

---

# STOP RULE

At the end of every phase, answer:

1. What did I discover?
2. What evidence supports it?
3. What decision did I make?
4. What assumptions did I make?
5. What could invalidate my conclusion?
6. What should I test next?

If you cannot answer these six questions, you are not finished with the phase.
